# 04 - Build Features

Name: Usha Oudeacoumar
Student ID: 20057009
Email: 20057009@mydbs.ie

This notebook turns the clean data from notebook 03 into numbers a model can actually
use. It converts the vitals columns to real numbers (NHAMCS stores them as text with
negative codes for missing), fills in missing values, and one-hot encodes the
category columns like sex and triage level. The reason for visit code (RFV1) has way
too many values so I just use the first 2 digits as a smaller group instead of the
whole code. The fill-in-missing-values and encoding steps are only fit on the
development years (2011-2014) so we are not cheating by looking at future years
data. Saves the final model-ready table and the fitted preprocessing pipeline.

In [12]:
import sys
sys.path.append(".")
import os
import pandas as pd
import numpy as np
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from utils import PROC_DIR, MODELS_DIR, load_json

os.makedirs(MODELS_DIR, exist_ok=True)

cfg = load_json("../configs/study_config.json")
dev_years = cfg["dev_years"]

df = pd.read_csv("../data/processed/clean_data.csv")
df.shape

(160526, 17)

In [13]:
# numeric columns (vitals + age), categorical columns (demographics + triage etc)
num_cols = ["AGE", "TEMPF", "PULSE", "RESPR", "BPSYS", "BPDIAS", "POPCT"]
cat_cols = ["SEX", "ETHIM", "RACER", "ARREMS", "IMMEDR", "PAINSCALE", "RESIDNCE"]

# nhamcs uses negative numbers for blank/unknown/not applicable, and also uses
# 998/999 as a blank code for some vitals, turn all of those into real missing
# values (NaN) instead of letting the model think they are real readings
vital_max = {
    "PULSE": 250,
    "RESPR": 60,
    "BPSYS": 250,
    "BPDIAS": 150,
}

def clean_numeric_col(col, max_ok=None):
    x = pd.to_numeric(col, errors="coerce")
    x = x.where(x >= 0)
    if max_ok is not None:
        x = x.where(x <= max_ok)
    return x

# TEMPF is stored as degrees F times 10 (981 means 98.1), fix that up first
df["TEMPF"] = clean_numeric_col(df["TEMPF"]) / 10
df["TEMPF"] = df["TEMPF"].where(df["TEMPF"] <= 110)

for c in ["PULSE", "RESPR", "BPSYS", "BPDIAS"]:
    df[c] = clean_numeric_col(df[c], vital_max.get(c))

df["AGE"] = clean_numeric_col(df["AGE"])
df["POPCT"] = clean_numeric_col(df["POPCT"], 100)

for c in cat_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")
    df.loc[df[c] < 0, c] = np.nan

df[num_cols + cat_cols].describe()

,AGE,TEMPF,PULSE,RESPR,BPSYS,BPDIAS,POPCT,SEX,ETHIM,RACER,ARREMS,IMMEDR,PAINSCALE,RESIDNCE
count,160526.000000,150711.000000,150022.000000,152389.000000,154269.000000,154022.000000,148627.000000,160526.000000,160526.000000,160526.000000,154614.000000,130430.000000,97801.000000,156007.000000
mean,46.605640,98.148429,85.941935,18.172598,136.692589,79.733752,97.168004,1.434472,1.865094,1.315444,1.810670,3.305513,5.099692,1.096970
std,19.799832,0.854759,17.615584,2.809862,23.302189,14.437328,6.048486,0.495689,0.341625,0.544348,0.391771,1.210737,3.666465,0.457626
min,18.000000,82.700000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000,0.000000,0.000000,1.000000
25%,30.000000,97.700000,74.000000,16.000000,121.000000,70.000000,97.000000,1.000000,2.000000,1.000000,2.000000,3.000000,0.000000,1.000000
50%,44.000000,98.100000,85.000000,18.000000,134.000000,79.000000,98.000000,1.000000,2.000000,1.000000,2.000000,3.000000,6.000000,1.000000
75%,61.000000,98.500000,96.000000,20.000000,149.000000,89.000000,99.000000,2.000000,2.000000,2.000000,2.000000,4.000000,8.000000,1.000000
max,94.000000,109.000000,236.000000,60.000000,250.000000,150.000000,100.000000,2.000000,2.000000,3.000000,2.000000,7.000000,10.000000,4.000000


In [14]:
# RFV1 (reason for visit) has hundreds of codes, too many for one hot encoding
# so just take the first 2 digits as a rough group (body system group)
df["rfv_group"] = df["RFV1"].astype(str).str.zfill(5).str[:2]
cat_cols = cat_cols + ["rfv_group"]

df["rfv_group"].value_counts().head(10)

rfv_group
19    24763
15    23813
10    23470
14    15410
12    12561
55     7219
11     6099
17     5233
58     4676
52     3859
Name: count, dtype: int64

In [15]:
# build the preprocessing pipeline: fill missing numbers with median then scale
# them (needed so logistic regression actually converges), fill missing
# categories with most common value, then one hot encode the categories
num_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])
cat_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocess = ColumnTransformer([
    ("num", num_pipe, num_cols),

    ("cat", cat_pipe, cat_cols),])

In [16]:
# only fit the pipeline on the development years, not the whole dataset, so we
# are not peeking at the future years when deciding fill-in values
dev_df = df[df["yr"].isin(dev_years)]
preprocess.fit(dev_df[num_cols + cat_cols])

feat_names = list(preprocess.get_feature_names_out())
len(feat_names)

90

In [17]:
# now apply the fitted pipeline to everything (dev years and eval years)
X = preprocess.transform(df[num_cols + cat_cols])
X_df = pd.DataFrame(X, columns=feat_names)

model_ready = pd.concat([df[["yr", "admit"]].reset_index(drop=True), X_df], axis=1)
model_ready.shape

(160526, 92)

In [18]:
# save the model ready table and the fitted pipeline so notebook 05 can use them
model_ready.to_csv(os.path.join(PROC_DIR, "model_ready.csv"), index=False)
joblib.dump(preprocess, os.path.join(MODELS_DIR, "preprocess_pipeline.pkl"))
print("saved model ready data and preprocessing pipeline")

saved model ready data and preprocessing pipeline
